# Word2Vec: CBOW vs Skip-Gram

Word2Vec trains a neural network to predict words from context, producing dense vector representations where semantically similar words cluster together. Two architectures exist: CBOW predicts a target word from surrounding context words, while Skip-Gram does the reverse. Both produce word embeddings but differ in training dynamics and performance characteristics — CBOW is faster and suits frequent words; Skip-Gram handles rare words better and generally produces richer representations on large corpora.


## Install and Import

In [2]:
!pip install gensim -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.2 MB/s eta 0:00:00


## Text Dataset

A small corpus covering animals, royalty, geography and nature. In production, replace this with Wikipedia, news archives, or domain-specific text — larger and more varied corpora produce significantly better embeddings.


In [3]:
import re
from gensim.models import Word2Vec

print("Ready.")


Ready.


In [4]:
raw_text = """
A pillow is a soft, cushioned object designed to support the head and neck during rest or sleep, playing an essential role in maintaining comfort and proper posture. Typically filled with materials such as cotton, foam, feathers, or synthetic fibers, pillows come in various shapes and sizes to suit different sleeping positions and personal preferences. Beyond sleep, pillows are also used for relaxation while sitting, reading, or watching television, providing added comfort and support. They contribute significantly to sleep quality, as the right pillow can help prevent neck pain, reduce strain, and promote better alignment of the spine. In addition to their functional use, pillows also serve decorative purposes in homes, enhancing the aesthetic appeal of beds, sofas, and living spaces with different colors, patterns, and textures.
"""

print(f"Corpus loaded.")


Corpus loaded.


## Preprocessing

Lowercase the text, strip punctuation, and split into sentences. Each sentence becomes a list of tokens — the format gensim expects.


In [5]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    sentences = []
    for line in text.strip().split('\n'):
        words = line.strip().split()
        if len(words) > 1:
            sentences.append(words)
    return sentences

sentences = preprocess(raw_text)
all_words = [w for s in sentences for w in s]

print(f"Sentences : {len(sentences)}")
print(f"Tokens    : {len(all_words)}")
print(f"Unique    : {len(set(all_words))}")
print(f"\nSample:")
for s in sentences[:3]:
    print(f"  {' '.join(s)}")


Sentences : 1
Tokens    : 127
Unique    : 95

Sample:
  a pillow is a soft cushioned object designed to support the head and neck during rest or sleep playing an essential role in maintaining comfort and proper posture typically filled with materials such as cotton foam feathers or synthetic fibers pillows come in various shapes and sizes to suit different sleeping positions and personal preferences beyond sleep pillows are also used for relaxation while sitting reading or watching television providing added comfort and support they contribute significantly to sleep quality as the right pillow can help prevent neck pain reduce strain and promote better alignment of the spine in addition to their functional use pillows also serve decorative purposes in homes enhancing the aesthetic appeal of beds sofas and living spaces with different colors patterns and textures


## Train Both Models

Key parameters: `vector_size` sets embedding dimensions; `window` controls context span; `min_count` ignores rare tokens; `sg=0` selects CBOW, `sg=1` selects Skip-Gram; `epochs` controls training passes.


In [7]:
cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=0,        # CBOW
    epochs=200,
    seed=42
)

sg_model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,        # Skip-Gram
    epochs=200,
    seed=42
)

print("Both models trained.")


Both models trained.


## Vocabulary Size

Both models share the same vocabulary since they train on the same corpus. Every unique token gets a 50-dimensional vector.


In [8]:
vocab = list(cbow_model.wv.key_to_index.keys())

print(f"Vocabulary size: {len(vocab)} words")
print(f"\nAll tokens:")
print(sorted(vocab))


Vocabulary size: 95 words

All tokens:
['a', 'added', 'addition', 'aesthetic', 'alignment', 'also', 'an', 'and', 'appeal', 'are', 'as', 'beds', 'better', 'beyond', 'can', 'colors', 'come', 'comfort', 'contribute', 'cotton', 'cushioned', 'decorative', 'designed', 'different', 'during', 'enhancing', 'essential', 'feathers', 'fibers', 'filled', 'foam', 'for', 'functional', 'head', 'help', 'homes', 'in', 'is', 'living', 'maintaining', 'materials', 'neck', 'object', 'of', 'or', 'pain', 'patterns', 'personal', 'pillow', 'pillows', 'playing', 'positions', 'posture', 'preferences', 'prevent', 'promote', 'proper', 'providing', 'purposes', 'quality', 'reading', 'reduce', 'relaxation', 'rest', 'right', 'role', 'serve', 'shapes', 'significantly', 'sitting', 'sizes', 'sleep', 'sleeping', 'sofas', 'soft', 'spaces', 'spine', 'strain', 'such', 'suit', 'support', 'synthetic', 'television', 'textures', 'the', 'their', 'they', 'to', 'typically', 'use', 'used', 'various', 'watching', 'while', 'with']


## Word Vectors

Each word is encoded as a list of 50 floats. The absolute values are not interpretable individually — what matters is the geometric relationship between vectors. Only the first 10 values are shown here.


In [10]:
a = ['pillow', 'sleep', 'comfort', 'head', 'neck']

print(f"{'Word':<10}  {'Model':<10}  Vector (first 10 values)")
print("-" * 72)

for b in a:
    for label, model in [("CBOW", cbow_model), ("Skip-Gram", sg_model)]:
        vec = model.wv[b]
        vals = ', '.join([f'{v:.3f}' for v in vec[:10]])
        print(f"{b:<10}  {label:<10}  [{vals} ...]")
    print()

Word        Model       Vector (first 10 values)
------------------------------------------------------------------------
pillow      CBOW        [0.044, -0.105, -0.061, -0.059, 0.060, 0.073, -0.033, 0.146, -0.224, -0.006 ...]
pillow      Skip-Gram   [0.075, -0.139, -0.056, -0.104, 0.104, 0.077, -0.001, 0.168, -0.287, 0.027 ...]

sleep       CBOW        [0.024, -0.130, -0.074, -0.121, 0.089, 0.099, -0.051, 0.206, -0.319, -0.022 ...]
sleep       Skip-Gram   [0.043, -0.092, -0.033, -0.158, 0.104, 0.088, 0.015, 0.177, -0.300, -0.011 ...]

comfort     CBOW        [0.038, -0.105, -0.043, -0.120, 0.088, 0.107, -0.060, 0.202, -0.259, -0.010 ...]
comfort     Skip-Gram   [0.056, -0.066, -0.003, -0.183, 0.106, 0.117, -0.009, 0.202, -0.271, 0.001 ...]

head        CBOW        [0.042, -0.106, -0.046, -0.074, 0.064, 0.066, -0.042, 0.143, -0.240, -0.010 ...]
head        Skip-Gram   [0.062, -0.116, -0.039, -0.111, 0.084, 0.068, -0.008, 0.147, -0.277, 0.030 ...]

neck        CBOW        [0.032, -0.110

## Top-5 Similar Words

Similarity is measured by cosine distance between vectors. The two models often agree on the top candidates but differ in ranking, reflecting their different training objectives.


In [11]:
new_vec = ['pillow', 'sleep', 'comfort', 'head', 'neck']

for x in new_vec:
    print(f"Query: '{x}'")
    print(f"  {'Rank':<5}  {'CBOW':<28}  {'Skip-Gram':<28}")
    print(f"  {'----':<5}  {'-'*26:<28}  {'-'*26:<28}")

    cbow_sim = cbow_model.wv.most_similar(x, topn=5)
    sg_sim   = sg_model.wv.most_similar(x, topn=5)

    for i, ((cw, cs), (sw, ss)) in enumerate(zip(cbow_sim, sg_sim), 1):
        cb = f"{cw:<14} {cs:.4f}"
        sk = f"{sw:<14} {ss:.4f}"
        print(f"  {i:<5}  {cb:<28}  {sk:<28}")
    print()


Query: 'pillow'
  Rank   CBOW                          Skip-Gram                   
  ----   --------------------------    --------------------------  
  1      reading        0.9950         to             0.9937       
  2      in             0.9946         neck           0.9935       
  3      beyond         0.9940         head           0.9932       
  4      the            0.9939         prevent        0.9930       
  5      and            0.9938         right          0.9930       

Query: 'sleep'
  Rank   CBOW                          Skip-Gram                   
  ----   --------------------------    --------------------------  
  1      and            0.9973         contribute     0.9957       
  2      or             0.9972         quality        0.9947       
  3      in             0.9972         support        0.9943       
  4      the            0.9967         or             0.9942       
  5      to             0.9966         significantly  0.9940       

Query: 'comfort

## Summary

**CBOW** averages context embeddings to predict a center word. It trains faster and performs well when the corpus is small or words are frequent.

**Skip-Gram** treats each context word as a separate training example. It takes longer to train but generalises better to rare words and typically produces higher-quality embeddings on large corpora.

Both models represent words as dense vectors in a shared geometric space. Distance and direction in that space encode semantic relationships — synonyms cluster, antonyms tend to oppose, and analogy tasks resolve to simple vector arithmetic.
